# A feature table without merge code

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toclocoinc/lattice-grid-notebooks/blob/main/retail-feature-table/retail_feature_table.ipynb)

Four DataFrames from four systems (customers, transactions, an entity/attribute/value table of custom attributes, churn labels) go into the Lattice Grid **data router**. It joins them, rolls the transactions up per customer, spreads the attributes into columns and turns text dates, decimal-comma amounts and Y/N flags into real dates, numbers and booleans. One live grid of customer features comes out, `widget.view` hands it to scikit-learn, and when the transactions change only the customers who bought something change. No keys, no `merge`, no `groupby`, no `pivot`.

**Data:** seeded synthetic retail data generated in the first cell below the install line (36 customers, 143 transactions, 144 attribute rows, 36 churn labels): the same seed and shape as the [web feature-table demo](https://toclocoinc.github.io/lattice-grid-demo-feature-table/). Every figure is invented; no real person or shop is described, so there is no data licence to note. The notebook code is MIT-licensed (see the repo `LICENSE`).

In [1]:
%pip install -q "lattice-grid-jupyter>=1.111.3.0"

## 1. Four untidy feeds

The way exported systems really look: dates are `dd/MM/yyyy` text, amounts are `"1.234,56"` strings, custom attributes are one row per value, churn is `Y`/`N`. A small seeded generator (a port of the web demo's, so the rows are identical) builds them as four pandas DataFrames.

In [2]:
import datetime as dt, math, pandas as pd

def make_feeds(seed=20261019):
    s = [seed]
    def rand():  # mulberry32, the generator the web demo uses
        s[0] = (s[0] + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((s[0] ^ (s[0] >> 15)) * (1 | s[0])) & 0xFFFFFFFF
        t = ((t + ((t ^ (t >> 7)) * (61 | t) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    pick = lambda xs: xs[int(rand() * len(xs))]
    day = lambda n: dt.date(2026, 1, 1) + dt.timedelta(days=n)
    dmy = lambda d: d.strftime("%d/%m/%Y")
    comma = lambda x: f"{x:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")
    first = "Amara Ben Chloe Dev Elif Farid Grace Hana Ivo Jun Kofi Lena Mateo Noor Omar Priya Quinn Rosa Sam Tara".split()
    last = "Adler Brook Chen Dalton Evans Fox Gill Hart Iyer Jones Khan Lowe Moss Nash Okoro Park".split()
    cust, txn, attr, churn = [], [], [], []
    for i in range(1, 37):
        cid = f"C{i:03d}"
        cust.append((cid, f"{first[i % 20]} {pick(last)}", pick(["North", "South", "East", "West"]), dmy(day(-int(rand() * 700)))))
        plan = pick(["Free", "Plus", "Plus", "Pro"])
        vals = {"plan": plan, "nps": str(int(rand() * 11)), "newsletter": pick(["yes", "no"]),
                "loyalty_since": day(-int(rand() * 500)).isoformat()}
        for a, v in vals.items():
            attr.append((f"A{len(attr) + 1:04d}", cid, a, v))
        orders = int(rand() * 9)
        for _ in range(orders):
            amount = math.floor((8 + rand() * rand() * 2400) * 100 + 0.5) / 100
            txn.append([f"T{len(txn) + 1:04d}", cid, comma(amount), dmy(day(int(rand() * 290)))])
        churn.append((f"L{i:03d}", cid, "Y" if orders < 2 and rand() < 0.7 else "N"))
    for i in range(len(txn) - 1, 0, -1):
        j = int(rand() * (i + 1)); txn[i], txn[j] = txn[j], txn[i]
    for i, t in enumerate(txn): t[0] = f"T{i + 1:04d}"
    return (pd.DataFrame(cust, columns=["id", "name", "region", "joined"]),
            pd.DataFrame(txn, columns=["id", "customer_id", "amount", "date"]),
            pd.DataFrame(attr, columns=["id", "customer_id", "attribute", "value"]),
            pd.DataFrame(churn, columns=["id", "customer_id", "churned"]))


customers, transactions, attributes, churn = make_feeds()
transactions.head(4)

,id,customer_id,amount,date
0,T0001,C004,"33,02",31/07/2026
1,T0002,C019,"981,50",12/08/2026
2,T0003,C026,"777,09",25/09/2026
3,T0004,C010,"20,19",22/06/2026


## 2. Declare the shape, not the code

`LatticeRouter` takes the four frames and a declaration. The customer source **collects** its transactions and rolls them up (count, sum of amount, date of the last purchase), **spreads** the attribute rows into `cf_plan`, `cf_nps`, ... columns, and looks up the churn label. Field specs coerce the strings as they arrive. The router runs in the grid's own data-router module, so the join semantics are the grid's, not a pandas reimplementation.

In [3]:
from lattice_grid_jupyter import LatticeRouter, LatticeGridWidget

DMY = {"type": "date", "format": "dd/MM/yyyy"}
router = LatticeRouter(
    sources={
        "customers": {"data": customers, "key": "id", "fields": {"joined": DMY}, "join": [
            {"from": "transactions", "foreign_key": "customer_id", "many": True, "aggregate": {
                "orders": {"fn": "count"}, "spend": {"fn": "sum", "field": "amount"},
                "last_purchase": {"fn": "last", "field": "date", "order_by": "date"}}},
            {"from": "attributes", "foreign_key": "customer_id", "spread": {
                "name": "attribute", "value": "value", "prefix": "cf_",
                "type": {"nps": "number", "newsletter": "boolean", "loyalty_since": "date"}}},
            {"from": "churn", "local_key": "id", "foreign_key": "customer_id", "fields": ["churned"]}]},
        "transactions": {"data": transactions, "key": "id", "fields": {
            "amount": {"type": "number", "decimal": ","}, "date": DMY}},
        "attributes": {"data": attributes, "key": "id"},
        "churn": {"data": churn, "key": "id", "fields": {
            "churned": {"type": "boolean", "true": ["Y"], "false": ["N"]}}},
    },
    routes={"features": "customers"})

widths = dict(id=54, name=112, region=70, joined=108, orders=64, spend=80, last_purchase=108, cf_plan=66,
              cf_nps=62, cf_newsletter=112, cf_loyalty_since=118, churned=74)
w = LatticeGridWidget(router, route="features", height=520,
                      columns=[{"field": f, "layout": {"width": x}, **({"format": {"decimals": 2}} if f == "spend" else {}),
                                  **({"type": "date"} if f in ("joined", "last_purchase", "cf_loyalty_since") else {})}
                               for f, x in widths.items()])
w

![the grid](https://raw.githubusercontent.com/toclocoinc/lattice-grid-notebooks/main/retail-feature-table/grid.png)

*The grid in JupyterLab: one row per customer, with the rollup columns (`orders`, `spend`, `last_purchase`), the spread attributes (`cf_*`) and the churn label, all derived by the router.*

## 3. The view goes straight into scikit-learn

`w.view` is the grid's rows as a typed DataFrame: the same table you see, in the grid's sort order and with its filters applied (sort or filter the grid and it follows). The rollups are numbers, the spread attributes arrive as typed columns (`cf_nps` a number, `cf_newsletter` a boolean), the churn label is a boolean, and the one date rollup (`last_purchase`) arrives as ISO text, so a single `pd.to_datetime` is the only cleaning before the model.

The view is computed in the kernel, so this cell works the same whether you run it by hand or with *Run All*.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

view = w.view
X = pd.DataFrame({
    "orders": view["orders"], "spend": view["spend"],
    "days_since_purchase": (pd.Timestamp("2026-10-19") - pd.to_datetime(view["last_purchase"])).dt.days,
    "nps": view["cf_nps"], "newsletter": view["cf_newsletter"].astype(int),
    "pro": (view["cf_plan"] == "Pro").astype(int)}).fillna(0)
y = view["churned"].astype(int)
auc = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=4, scoring="roc_auc").mean()
print(f"{len(view)} customers, {y.sum()} churned; cross-validated ROC AUC {auc:.2f}")
X.head()

36 customers, 5 churned; cross-validated ROC AUC 0.87


,orders,spend,days_since_purchase,nps,newsletter,pro
0,8,4911.38,66.0,10,0,0
1,3,1809.17,135.0,2,0,0
2,3,1539.90,30.0,0,0,1
3,2,165.51,80.0,9,1,1
4,6,3443.17,65.0,8,0,0


## 4. Change a source, and only the affected rows change

Three new transactions arrive for three customers. `router.update` sends the grid a **keyed diff** of the transactions source, not a reload, and the router re-derives only the customers those rows belong to (their cells flash in the grid above). This cell sends the diff; the next one compares the view before and after.

In [5]:
before = w.view.set_index("id")
new = pd.DataFrame({"id": ["T9001", "T9002", "T9003"], "customer_id": ["C003", "C011", "C017"],
                    "amount": ["1.250,00", "89,90", "430,50"], "date": ["19/10/2026"] * 3})
router.update("transactions", pd.concat([transactions, new], ignore_index=True))

{'added': 3, 'updated': 0, 'removed': 0}

In [6]:
after = w.view.set_index("id").reindex(before.index)
changed = ((before != after) & ~(before.isna() & after.isna())).any(axis=1)
print(f"{changed.sum()} of {len(after)} customers changed: {', '.join(changed[changed].index)}")
after.loc[changed, ["name", "orders", "spend", "last_purchase"]].assign(
    orders_before=before["orders"], spend_before=before["spend"])

3 of 36 customers changed: C003, C011, C017


,name,orders,spend,last_purchase,orders_before,spend_before
id,,,,,,
C003,Dev Park,4,2789.90,2026-10-19,3,1539.90
C011,Lena Nash,2,441.61,2026-10-19,1,351.71
C017,Rosa Adler,9,7155.78,2026-10-19,8,6725.28
